In [ ]:
from pathlib import Path

root = Path("/content/_MusicProject")
data_dir = root / "data"
data_dir.mkdir(parents=True, exist_ok=True)

from google.colab import files
uploaded = files.upload()

Saving yamnet_embeddings.npz to yamnet_embeddings.npz


In [ ]:
import shutil

shutil.move("yamnet_embeddings.npz", data_dir / "yamnet_embeddings.npz")
!ls -R /content/_MusicProject

/content/_MusicProject:
data

/content/_MusicProject/data:
yamnet_embeddings.npz


In [ ]:
import numpy as np
from pathlib import Path

root = Path("/content/_MusicProject")
data_path = root / "data" / "yamnet_embeddings.npz"

data = np.load(data_path, allow_pickle=True)
print(data.files)

X = data["X"]
y = data["y"]
paths = data["paths"]

print("[INFO] X shape:", X.shape)
print("[INFO] y shape:", y.shape)
print("[INFO] num study:", np.sum(y == 1))
print("[INFO] num general:", np.sum(y == 0))

['X', 'y', 'paths']
[INFO] X shape: (236, 1024)
[INFO] y shape: (236,)
[INFO] num study: 122
[INFO] num general: 114


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_temp, y_train, y_temp, paths_train, paths_temp = train_test_split(
    X, y, paths, test_size=0.3, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test, paths_val, paths_test = train_test_split(
    X_temp, y_temp, paths_temp, test_size=0.5, random_state=42, stratify=y_temp
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("[INFO] Train size:", X_train_scaled.shape[0])
print("[INFO] Val size:", X_val_scaled.shape[0])
print("[INFO] Test size:", X_test_scaled.shape[0])

[INFO] Train size: 165
[INFO] Val size: 35
[INFO] Test size: 36


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

X_train_tf = X_train_scaled.astype("float32")
X_val_tf = X_val_scaled.astype("float32")
X_test_tf = X_test_scaled.astype("float32")

y_train_tf = y_train.astype("float32")
y_val_tf = y_val.astype("float32")
y_test_tf = y_test.astype("float32")

input_dim = X_train_tf.shape[1]

inputs = keras.Input(shape=(input_dim,))
x = layers.Dense(256, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-4))(inputs)
x = layers.Dropout(0.3)(x)
x = layers.Dense(64, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-4))(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", keras.metrics.AUC(name="auc")]
)

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_auc",
    mode="max",
    patience=8,
    restore_best_weights=True
)

history = model.fit(
    X_train_tf,
    y_train_tf,
    validation_data=(X_val_tf, y_val_tf),
    epochs=80,
    batch_size=16,
    verbose=2,
    callbacks=[early_stop]
)

Epoch 1/80
11/11 - 8s - 769ms/step - accuracy: 0.8242 - auc: 0.9081 - loss: 0.4855 - val_accuracy: 0.9143 - val_auc: 0.9592 - val_loss: 0.7055
Epoch 2/80
11/11 - 0s - 36ms/step - accuracy: 0.9394 - auc: 0.9714 - loss: 0.4847 - val_accuracy: 0.9143 - val_auc: 0.9641 - val_loss: 0.3982
Epoch 3/80
11/11 - 0s - 38ms/step - accuracy: 0.9576 - auc: 0.9681 - loss: 0.2390 - val_accuracy: 0.9143 - val_auc: 0.9624 - val_loss: 1.4752
Epoch 4/80
11/11 - 1s - 64ms/step - accuracy: 0.9636 - auc: 0.9885 - loss: 0.4425 - val_accuracy: 0.9143 - val_auc: 0.9641 - val_loss: 0.8348
Epoch 5/80
11/11 - 0s - 26ms/step - accuracy: 0.9697 - auc: 0.9893 - loss: 0.1898 - val_accuracy: 0.9143 - val_auc: 0.9641 - val_loss: 0.5331
Epoch 6/80
11/11 - 0s - 10ms/step - accuracy: 0.9697 - auc: 0.9918 - loss: 0.1991 - val_accuracy: 0.9143 - val_auc: 0.9657 - val_loss: 0.5046
Epoch 7/80
11/11 - 0s - 9ms/step - accuracy: 0.9879 - auc: 0.9931 - loss: 0.2594 - val_accuracy: 0.9143 - val_auc: 0.9657 - val_loss: 0.5317
Epoch 

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

test_probs = model.predict(X_test_tf).ravel()
test_preds = (test_probs >= 0.5).astype(int)

test_acc = accuracy_score(y_test_tf, test_preds)
test_auc = roc_auc_score(y_test_tf, test_probs)

print("[TEST] Accuracy:", round(test_acc, 3))
print("[TEST] ROC-AUC:", round(test_auc, 3))
print("\n[TEST] Classification report:\n")
print(classification_report(y_test_tf, test_preds))

2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 417ms/step
[TEST] Accuracy: 0.944
[TEST] ROC-AUC: 0.988

[TEST] Classification report:

              precision    recall  f1-score   support

         0.0       0.94      0.94      0.94        17
         1.0       0.95      0.95      0.95        19

    accuracy                           0.94        36
   macro avg       0.94      0.94      0.94        36
weighted avg       0.94      0.94      0.94        36



In [ ]:
from pathlib import Path
import numpy as np

root = Path("/content/_MusicProject")
models_dir = root / "models_yamnet"
models_dir.mkdir(parents=True, exist_ok=True)

model_path = models_dir / "focus_yamnet_tf.keras"
scaler_path = models_dir / "focus_yamnet_scaler.npy"

model.save(model_path)
np.save(scaler_path, {"mean": scaler.mean_, "scale": scaler.scale_})

print("[INFO] Saved:")
print(model_path)
print(scaler_path)

[INFO] Saved:
/content/_MusicProject/models_yamnet/focus_yamnet_tf.keras
/content/_MusicProject/models_yamnet/focus_yamnet_scaler.npy


In [ ]:
from google.colab import files

files.download(str(model_path))
files.download(str(scaler_path))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>